# False Assurance — held-out study

Open in Google Colab with a GPU and choose **Run all**. The configuration is set for
the held-out study. Return the single results ZIP offered by the last cell.

The study uses 256 Integrated Gradients points, both architectures, and seeds 11, 29, and
47. It trains all six checkpoint pairs and checks each on development data first. If any
pair fails, it exports the development diagnostics for review. Once all six pass, the
notebook calibrates the audit thresholds and compares missed-risk rates on the held-out
test split.


In [ ]:
from collections import defaultdict
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import random
import shutil
import tarfile
import zipfile
from time import perf_counter
from urllib.request import urlopen

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageOps, __version__ as pillow_version
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
from torchvision import models

MODE = "study"  # Use "smoke" for a synthetic check.
STUDY_FROZEN = True
PILOT_ARCH = "mobilenet_v3_large"  # Repeat pilot with "resnet34" after initial checks.
PILOT_SEED = 11
PILOT_EPOCHS = 12
STUDY_EPOCHS = 12
STUDY_ARCHS = ("mobilenet_v3_large", "resnet34")
STUDY_SEEDS = (11, 29, 47)
SPLIT_SEED = 1729
IMAGE_SIZE = 112
PATCH_TOP, PATCH_LEFT, PATCH_SIZE = 5, 5, 20
ALPHA = 0.90
TRAIN_ALIGNMENT = 0.90
BLUR_RADIUS = 4
OCCLUSION_SIZE = 48
MARGIN_DROP = 1.0
IG_STEPS = 256
IG_CHUNK = 4
AUDIT_BATCH = 16
TRAIN_BATCH = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
CAL_FALSE_FLAG_RATE = 0.05
N_BOOT = 2000
N_DEV_PER_CLASS = 72
N_CAL_PER_CLASS = 96
CLASSES = {"n02102040": 0, "n03417042": 1}
CLASS_NAMES = {0: "English springer", 1: "garbage truck"}
URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
ARCHIVE_SHA256 = "64d0c4859f35a461889e0147755a999a48b49bf38a7e0f9bd27003f10db02fe5"
PROTOCOL_VERSION = "0.3-study"
REFRESH_BATCH_NORM = True
ARCHS = ("mobilenet_v3_large", "resnet34")
DEV_SCORE_LIMIT_PER_CLASS = 24
DEV_MIN_RISK = 20
DEV_MIN_STRESS_ONLY_RISK = 10
DEV_MIN_ALIGNED_ACCURACY = 0.80
DEV_MIN_CONTROL_NOMINAL_ACCURACY = 0.75
DEV_MAX_CONTROL_STRESS_RISK = 0.10
DEV_MAX_IG_P90_RESIDUAL = 0.20
POLICIES = {
    "nominal_only": ("nominal", "dim", "bright"),
    "context_diverse": ("nominal", "blur", "occlusion"),
}
ORACLE_CONTEXTS = ("nominal", "blur", "occlusion")
METHODS = ("gradcam", "ig", "neutralize")
PATCH = (slice(PATCH_TOP, PATCH_TOP + PATCH_SIZE), slice(PATCH_LEFT, PATCH_LEFT + PATCH_SIZE))

assert MODE in {"smoke", "pilot", "study"} and PILOT_ARCH in ARCHS
assert MODE != "study" or STUDY_FROZEN, "Freeze and log the protocol before running study."
assert STUDY_ARCHS and set(STUDY_ARCHS) <= set(ARCHS) and STUDY_SEEDS
assert all(len(views) == 3 for views in POLICIES.values())
assert PATCH_SIZE % 2 == 0 and PATCH_TOP + PATCH_SIZE <= (IMAGE_SIZE - OCCLUSION_SIZE) // 2
assert IG_STEPS >= 2 and IG_CHUNK >= 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(6, os.cpu_count() or 1))
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

BASE = Path.cwd() / "false_assurance"
BASE.mkdir(exist_ok=True)
OUT = BASE / f"run_{MODE}_{os.urandom(4).hex()}"
OUT.mkdir()
print(f"{MODE=} | {DEVICE=} | PyTorch {torch.__version__} | {OUT}")
if DEVICE.type == "cpu" and MODE != "smoke":
    print("CPU runtime selected. Training and attribution can take substantially longer.")


def save_json(path, data):
    path.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def file_sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
if MODE == "study":
    assert (STUDY_ARCHS, STUDY_SEEDS, IG_STEPS) == (ARCHS, (11, 29, 47), 256), (
        "Use both architectures, all three seeds, and 256 IG points for the held-out study."
    )


# Build the split and render the interventions

Official training images supply the models. The official validation images are split by class into development, calibration and test. Exact duplicates after deterministic resizing are handled before splitting and logged. The dataset manifest records the exact source filenames and processed hashes. Smoke mode substitutes generated fixtures.

Marker templates use the same corner, opacity and grayscale histogram. They differ in orientation. The gray neutral template shares the same support and mean intensity. The renderer applies context before blending, preserving the template and opacity across contexts. Central occlusion leaves the patch area intact.

In [ ]:
def read_data():
    arrays, rows = [], []
    if MODE == "smoke":
        rng = np.random.default_rng(SPLIT_SEED)
        for origin, count in (("train", 12), ("val", 12)):
            for y in CLASSES.values():
                for i in range(count):
                    image = rng.integers(20, 220, (IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
                    image[35:80, 35:80, y] = 240
                    arrays.append(image)
                    rows.append((f"{origin}/{y}/{i}", y, origin))
        return arrays, rows, None

    archive = BASE / "imagenette2-160.tgz"
    root = BASE / "imagenette2-160"
    if not archive.exists():
        partial = archive.with_suffix(f".{OUT.name}.part")
        with urlopen(URL, timeout=60) as response, partial.open("wb") as stream:
            shutil.copyfileobj(response, stream)
        if file_sha256(partial) != ARCHIVE_SHA256:
            raise ValueError(f"Unexpected Imagenette SHA-256 for {partial}")
        partial.replace(archive)
    digest = file_sha256(archive)
    if digest != ARCHIVE_SHA256:
        raise ValueError(f"Unexpected Imagenette SHA-256 for {archive}")
    if not root.exists():
        staging = BASE / f"extract_{OUT.name}"
        staging.mkdir()
        with tarfile.open(archive, "r:gz") as dataset:
            chosen = []
            for member in dataset:
                parts = Path(member.name).parts
                if (member.isfile() and len(parts) == 4 and parts[0] == root.name
                        and parts[1] in ("train", "val") and parts[2] in CLASSES):
                    chosen.append(member)
            assert chosen, "Expected Imagenette archive layout."
            dataset.extractall(staging, chosen, filter="data")
        (staging / root.name).replace(root)
        staging.rmdir()
    for origin in ("train", "val"):
        for wnid, y in CLASSES.items():
            for path in sorted((root / origin / wnid).iterdir()):
                with Image.open(path) as img:
                    rgb = ImageOps.exif_transpose(img).convert("RGB")
                    image = ImageOps.fit(rgb, (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
                    arrays.append(np.array(image))
                    rows.append((path.relative_to(root).as_posix(), y, origin))
    return arrays, rows, digest


def split_data(arrays, rows):
    dev_n, cal_n = (4, 4) if MODE == "smoke" else (N_DEV_PER_CLASS, N_CAL_PER_CLASS)
    buckets = defaultdict(list)
    for i, image in enumerate(arrays):
        digest = hashlib.sha256(image.tobytes()).hexdigest()
        buckets[digest].append(i)
    index, exclusions = [], []
    for digest, group in buckets.items():
        if len({rows[i][1] for i in group}) > 1:
            exclusions.extend((rows[i][0], digest, "conflicting_labels") for i in group)
            continue
        index.append((*rows[group[0]], digest, group[0]))
        exclusions.extend((rows[i][0], digest, "duplicate_processed_pixels") for i in group[1:])
    frame = pd.DataFrame(index, columns=["image_id", "y", "official_split", "sha256", "array_id"])
    frame["split"] = "train"
    rng = np.random.default_rng(SPLIT_SEED)
    for y in CLASSES.values():
        val = frame.index[(frame.y == y) & (frame.official_split == "val")].to_numpy()
        assert len(val) > dev_n + cal_n
        val = rng.permutation(val)
        frame.loc[val[:dev_n], "split"] = "dev"
        frame.loc[val[dev_n:dev_n + cal_n], "split"] = "cal"
        frame.loc[val[dev_n + cal_n:], "split"] = "test"
    assert frame.sha256.is_unique and frame.image_id.is_unique
    pd.DataFrame(exclusions, columns=["image_id", "sha256", "reason"]).to_csv(
        OUT / "duplicate_exclusions.csv", index=False)
    return frame


def marker_codes(labels, seed, arm):
    rng = np.random.default_rng(seed + 1000)
    codes = labels.copy()
    for y in CLASSES.values():
        indices = rng.permutation(np.flatnonzero(labels == y))
        flip_fraction = (1 - TRAIN_ALIGNMENT) if arm == "shortcut" else 0.5
        flips = indices[:round(flip_fraction * len(indices))]
        codes[flips] = 1 - y
    return codes


def make_marker(code):
    marker = np.full((PATCH_SIZE, PATCH_SIZE, 1), 127.5, dtype=np.float32)
    if code == 0:
        marker[:PATCH_SIZE // 2, :, :] = 235
        marker[PATCH_SIZE // 2:, :, :] = 20
    if code == 1:
        marker[:, :PATCH_SIZE // 2, :] = 235
        marker[:, PATCH_SIZE // 2:, :] = 20
    return marker


MARKERS = np.stack([make_marker(code) for code in (0, 1, 2)])


def render(images, codes):
    pixels = images.astype(np.float32)
    patch = (slice(None), *PATCH, slice(None))
    pixels[patch] = (1 - ALPHA) * pixels[patch] + ALPHA * MARKERS[codes]
    return torch.from_numpy(pixels).permute(0, 3, 1, 2).contiguous().div_(127.5).sub_(1)


def contextualize(images, view):
    if view == "nominal":
        return images
    if view in ("dim", "bright"):
        factor = 0.95 if view == "dim" else 1.05
        return np.rint(np.clip(images.astype(np.float32) * factor, 0, 255)).astype(np.uint8)
    if view == "blur":
        return np.stack([np.array(Image.fromarray(img).filter(ImageFilter.GaussianBlur(BLUR_RADIUS)))
                         for img in images])
    if view == "occlusion":
        masked = images.copy()
        start = (IMAGE_SIZE - OCCLUSION_SIZE) // 2
        stop = start + OCCLUSION_SIZE
        masked[:, start:stop, start:stop] = 128
        return masked
    raise ValueError(view)


def materialize(frame):
    return np.stack([ARRAYS[i] for i in frame.array_id]), frame.y.to_numpy(dtype=np.int64, copy=True)


ARRAYS, raw_rows, archive_hash = read_data()
SPLIT = split_data(ARRAYS, raw_rows)
SPLIT.drop(columns="array_id").to_csv(OUT / "split_manifest.csv", index=False)
print(SPLIT.groupby(["split", "y"]).size().unstack())

In [ ]:
def make_model(arch):
    if arch == "resnet34":
        model = models.resnet34(weights=None, num_classes=2)
        return model, model.layer4[-1]
    model = models.mobilenet_v3_large(weights=None, num_classes=2)
    return model, model.features[-1]


@torch.no_grad()
def refresh_batch_norm(model, x, seed):
    """Recompute inference statistics with the training images and frozen weights."""
    model.eval()
    layers = [(layer, layer.momentum) for layer in model.modules() if isinstance(layer, nn.BatchNorm2d)]
    order = torch.randperm(len(x), generator=torch.Generator().manual_seed(seed))
    try:
        for layer, _ in layers:
            layer.reset_running_stats()
            layer.momentum = None
            layer.train()
        batch_size = 8 if MODE == "smoke" else TRAIN_BATCH
        for indices in order.split(batch_size):
            model(x[indices].to(DEVICE))
    finally:
        for layer, momentum in layers:
            layer.momentum = momentum
        model.eval()


def train_model(arch, seed, arm, epochs, folder):
    set_seed(seed)
    model, layer = make_model(arch)
    model.to(DEVICE)
    frame = SPLIT[SPLIT.split == "train"]
    images, labels = materialize(frame)
    codes = marker_codes(labels, seed, arm)
    frame[["image_id", "y"]].assign(marker_code=codes).to_csv(
        folder / f"{arm}_training_markers.csv", index=False)
    data = TensorDataset(render(images, codes), torch.from_numpy(labels))
    loader = DataLoader(data, batch_size=8 if MODE == "smoke" else TRAIN_BATCH,
                        shuffle=True, generator=torch.Generator().manual_seed(seed))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    metrics = []
    for epoch in range(epochs):
        start = perf_counter()
        model.train()
        total_loss, total_correct = 0.0, 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
            total_correct += (logits.argmax(1) == y).sum().item()
        metrics.append({"epoch": epoch + 1, "loss": total_loss / len(data),
                        "accuracy": total_correct / len(data), "seconds": perf_counter() - start})
        scheduler.step()
        print(arch, seed, arm, metrics[-1], flush=True)
    pd.DataFrame(metrics).to_csv(folder / f"{arm}_training.csv", index=False)
    if REFRESH_BATCH_NORM:
        refresh_batch_norm(model, data.tensors[0], seed)
    torch.save(model.state_dict(), folder / f"{arm}.pt")
    model.eval().requires_grad_(False)
    return model, layer

# Keep behavioral risk separate from audit evidence

A risky view requires a correct aligned prediction, an incorrect identity-swapped prediction and a decrease of at least 1.0 in the true-class logit margin. The primary case-level risk is the union over blur and central occlusion. The nominal swap is a diagnostic. A switch isolates the model response to this particular marker edit on a fixed base image.

Audit scores use the margin for the class predicted on each aligned view. Grad-CAM uses positive heatmap mass inside the known patch after bilinear upsampling. Integrated Gradients uses absolute per-channel attribution mass with a mid-gray image baseline and Gauss–Legendre integration. Its signed completeness residual is tracked separately. Neutralization uses the absolute change in the fixed predicted-class margin when the two-tone code is replaced by gray.

In [ ]:
def class_margin(logits, target):
    return (logits[:, 1] - logits[:, 0]) * (2 * target - 1)


def patch_mass(heatmap):
    local_mass = heatmap[:, PATCH[0], PATCH[1]].sum((1, 2))
    total_mass = heatmap.sum((1, 2))
    return local_mass / total_mass.clamp_min(1e-12), total_mass <= 1e-12


def gradcam(model, layer, x):
    outputs = []
    hook = layer.register_forward_hook(lambda _, inputs, output: outputs.append(output))
    try:
        with torch.enable_grad():
            logits = model(x.detach().requires_grad_(True))
            target = logits.detach().argmax(1)
            features, = outputs
            gradient, = torch.autograd.grad(class_margin(logits, target).sum(), features)
    finally:
        hook.remove()
    weights = gradient.mean(dim=(2, 3), keepdim=True)
    cam = (weights * features.detach()).sum(1, keepdim=True).relu()
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    return patch_mass(cam)


def integrated_attribution(model, x, baseline, target, steps=IG_STEPS):
    nodes, weights = np.polynomial.legendre.leggauss(steps)
    nodes = x.new_tensor((nodes + 1) / 2)
    weights = x.new_tensor(weights / 2)
    delta = x - baseline
    summed = torch.zeros_like(x)
    with torch.enable_grad():
        for a, w in zip(nodes.split(IG_CHUNK), weights.split(IG_CHUNK), strict=True):
            path = (baseline[None] + a[:, None, None, None, None] * delta[None])
            path = path.flatten(0, 1).detach().requires_grad_(True)
            margin = class_margin(model(path), target.repeat(len(a)))
            gradient, = torch.autograd.grad(margin.sum(), path)
            summed += (gradient.reshape(len(a), *x.shape) * w[:, None, None, None, None]).sum(0)
    return delta * summed


def integrated_gradients(model, x):
    baseline = torch.zeros_like(x)
    with torch.no_grad():
        logits = model(x)
        target = logits.argmax(1)
        exact = class_margin(logits, target) - class_margin(model(baseline), target)
    attribution = integrated_attribution(model, x, baseline, target)
    residual = (attribution.sum((1, 2, 3)) - exact).abs() / exact.abs().clamp_min(1)
    score, empty = patch_mass(attribution.abs().sum(1))
    return score, empty, residual


@torch.no_grad()
def neutralize(model, x, neutral):
    logits = model(x)
    target = logits.argmax(1)
    return (class_margin(logits, target) - class_margin(model(neutral), target)).abs()


@torch.no_grad()
def predict(model, batch):
    return torch.cat([model(x.to(DEVICE)).cpu() for x in batch.split(AUDIT_BATCH)])


def oracle(model, frame, arm):
    images, labels = materialize(frame)
    y = torch.from_numpy(labels)
    rows = []
    for view in ORACLE_CONTEXTS:
        context = contextualize(images, view)
        aligned = predict(model, render(context, labels))
        swapped = predict(model, render(context, 1 - labels))
        neutral = predict(model, render(context, np.full_like(labels, 2)))
        a_ok = aligned.argmax(1) == y
        s_ok = swapped.argmax(1) == y
        drop = class_margin(aligned, y) - class_margin(swapped, y)
        rows.append(frame[["image_id", "y"]].assign(
            arm=arm, view=view, aligned_correct=a_ok.numpy(), swapped_correct=s_ok.numpy(),
            neutral_correct=(neutral.argmax(1) == y).numpy(), margin_drop=drop.numpy(),
            risky=(a_ok & ~s_ok & (drop >= MARGIN_DROP)).numpy(),
            error_corrected=(~a_ok & s_ok).numpy(),
        ))
    result = pd.concat(rows, ignore_index=True)
    assert np.isfinite(result.margin_drop).all()
    return result


AUDIT_COST = {"gradcam": (1, 1), "ig": (IG_STEPS + 2, IG_STEPS), "neutralize": (2, 0)}


def score_views(model, layer, frame, arm):
    images, labels = materialize(frame)
    rows, timings = [], []
    for view in dict.fromkeys(view for views in POLICIES.values() for view in views):
        context = contextualize(images, view)
        aligned = render(context, labels)
        neutral = render(context, np.full_like(labels, 2))
        for method in METHODS:
            sync()
            start = perf_counter()
            chunks, zeros, residuals = [], [], []
            for xb, nb in zip(aligned.split(AUDIT_BATCH), neutral.split(AUDIT_BATCH), strict=True):
                x = xb.to(DEVICE)
                if method == "gradcam":
                    score, empty = gradcam(model, layer, x)
                    zeros.append(empty.cpu())
                elif method == "ig":
                    score, empty, residual = integrated_gradients(model, x)
                    zeros.append(empty.cpu())
                    residuals.append(residual.cpu())
                else:
                    score = neutralize(model, x, nb.to(DEVICE))
                chunks.append(score.detach().cpu())
            sync()
            seconds = perf_counter() - start
            values = torch.cat(chunks).numpy()
            assert np.isfinite(values).all()
            rows.append(frame[["image_id", "y"]].assign(
                arm=arm, view=view, method=method, score=values,
                empty_map=torch.cat(zeros).numpy().astype(float) if zeros else np.nan,
                ig_residual=torch.cat(residuals).numpy() if residuals else np.nan,
            ))
            forwards, backwards = AUDIT_COST[method]
            timings.append({"arm": arm, "view": view, "method": method, "n_cases": len(frame),
                            "forward_images": forwards * len(frame),
                            "backward_images": backwards * len(frame), "seconds": seconds})
    return pd.concat(rows, ignore_index=True), pd.DataFrame(timings)


def stress_risk(truth):
    return truth[truth.view != "nominal"].groupby("image_id").risky.any().rename("risky")


def summarize_oracle(truth):
    return truth.groupby(["arm", "view"]).agg(
        n=("risky", "size"), n_risky=("risky", "sum"), n_corrected=("error_corrected", "sum"),
        aligned_accuracy=("aligned_correct", "mean"), swapped_accuracy=("swapped_correct", "mean"),
        neutral_accuracy=("neutral_correct", "mean"), median_margin_drop=("margin_drop", "median"),
    ).reset_index()

# Calibrate an audit decision and estimate missed risk

Each policy scores three views of every base case. Its decision uses the maximum score, with a separate threshold for each method, policy and checkpoint pair. The threshold is selected on the corresponding control checkpoint's calibration images at a target 5% empirical flag rate, using a strict exceedance rule to handle ties. Report realized flag rates on held-out controls and separately on cases without oracle-defined harm.

Missed-risk rate is the fraction of oracle-positive cases with no flag. A zero-risk denominator produces an undefined rate. The primary contrast is nominal-only minus context-diverse missed-risk rate for Grad-CAM and Integrated Gradients. Bootstrap intervals resample paired base images within each label; those intervals condition on the trained checkpoint and frozen threshold. Architecture and seed variation is reported separately.

In [ ]:
def portfolio_scores(scores):
    rows = []
    for policy, views in POLICIES.items():
        subset = scores[scores.view.isin(views)]
        grouped = subset.groupby(["image_id", "y", "arm", "method"]).score
        assert grouped.size().eq(len(views)).all()
        rows.append(grouped.max().rename("score").reset_index().assign(policy=policy))
    return pd.concat(rows, ignore_index=True)


def calibrate(scores):
    assert set(scores.arm) == {"control"}
    rows = []
    for (method, policy), items in portfolio_scores(scores).groupby(["method", "policy"]):
        values = np.sort(items.score.to_numpy())
        allowed = math.floor(len(values) * CAL_FALSE_FLAG_RATE)
        threshold = float(values[len(values) - allowed - 1])
        rows.append({"method": method, "policy": policy, "threshold": threshold,
                     "calibration_n": len(values), "calibration_flag_rate": (values > threshold).mean()})
    return pd.DataFrame(rows)


def make_decisions(scores, truth, thresholds):
    portfolio = portfolio_scores(scores)
    decisions = portfolio.merge(thresholds, on=["method", "policy"], validate="many_to_one")
    risks = truth[truth.view != "nominal"].groupby(["arm", "image_id"]).risky.any().reset_index()
    decisions = decisions.merge(risks, on=["arm", "image_id"], validate="many_to_one")
    nominal = truth[truth.view == "nominal"][["arm", "image_id", "risky"]].rename(
        columns={"risky": "nominal_risky"})
    decisions = decisions.merge(nominal, on=["arm", "image_id"], validate="many_to_one")
    assert len(decisions) == len(portfolio)
    decisions["flag"] = decisions.score > decisions.threshold
    return decisions


def ratio(numerator, denominator):
    return numerator / denominator if denominator else np.nan


def summarize_decisions(decisions):
    rows = []
    for (arm, method, policy), group in decisions.groupby(["arm", "method", "policy"]):
        risk, flag = group.risky, group.flag
        n_risky = int(risk.sum())
        n_missed = int((risk & ~flag).sum())
        rows.append({"arm": arm, "method": method, "policy": policy, "n": len(group),
                     "n_risky": n_risky, "n_missed": n_missed,
                     "missed_risk_rate": ratio(n_missed, n_risky), "flag_rate": flag.mean(),
                     "oracle_negative_flag_rate": flag[~risk].mean()})
    return pd.DataFrame(rows)


def paired_contrasts(decisions, seed):
    shortcut = decisions[decisions.arm == "shortcut"]
    rows = []
    for method, group in shortcut.groupby("method"):
        wide = group.pivot(index=["image_id", "y", "risky"], columns="policy", values="flag").reset_index()
        risk = wide.risky.to_numpy()
        nominal = wide.nominal_only.to_numpy(dtype=bool)
        diverse = wide.context_diverse.to_numpy(dtype=bool)
        # Difference in misses: positive when the diverse policy catches more cases.
        difference = diverse.astype(int) - nominal.astype(int)
        row = {"method": method, "n_risky": int(risk.sum()),
               "diverse_only_flags_among_risky": int((risk & diverse & ~nominal).sum()),
               "nominal_only_flags_among_risky": int((risk & nominal & ~diverse).sum()),
               "delta_mrr": np.nan, "bootstrap_low": np.nan, "bootstrap_high": np.nan}
        if risk.any():
            rng = np.random.default_rng(seed)
            strata = [np.flatnonzero(wide.y.to_numpy() == y) for y in CLASSES.values()]
            draws = np.concatenate([rng.choice(s, (N_BOOT, len(s))) for s in strata], axis=1)
            denominators = risk[draws].sum(1)
            usable = denominators > 0
            replicates = (difference[draws] * risk[draws]).sum(1)[usable] / denominators[usable]
            row["delta_mrr"] = float(difference[risk].mean())
            if len(replicates):
                row["bootstrap_low"], row["bootstrap_high"] = np.quantile(replicates, (0.025, 0.975))
        rows.append(row)
    return pd.DataFrame(rows)


def risk_sensitivity(decisions, truth, seed):
    tables = []
    for cutoff in sorted({0.5, MARGIN_DROP, 2.0}):
        for context in ("any_stress", *ORACLE_CONTEXTS[1:]):
            views = truth[truth.view.isin(ORACLE_CONTEXTS[1:])] if context == "any_stress" else (
                truth[truth.view == context])
            risky = views.aligned_correct & ~views.swapped_correct & (views.margin_drop >= cutoff)
            risk = views.assign(risky=risky).groupby(["arm", "image_id"]).risky.any().reset_index()
            scored = decisions.drop(columns="risky").merge(
                risk, on=["arm", "image_id"], validate="many_to_one")
            tables.append(paired_contrasts(scored, seed).assign(cutoff=cutoff, context=context))
    return pd.concat(tables, ignore_index=True)

In [ ]:
def implementation_checks():
    frame = SPLIT[SPLIT.split == "dev"].head(2)
    images, labels = materialize(frame)
    keep = np.ones((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)
    keep[PATCH] = False
    for view in ORACLE_CONTEXTS:
        context = contextualize(images, view)
        aligned = render(context, labels)
        swapped = render(context, 1 - labels)
        neutral = render(context, np.full_like(labels, 2))
        assert torch.equal(aligned[:, :, keep], swapped[:, :, keep])
        assert torch.equal(aligned[:, :, keep], neutral[:, :, keep])
        assert (aligned != swapped).flatten(1).any(1).all()
        torch.testing.assert_close((aligned + swapped).sum((2, 3)), 2 * neutral.sum((2, 3)),
                                   atol=2e-3, rtol=1e-5)

    linear = nn.Sequential(nn.Flatten(), nn.Linear(3 * 4 * 4, 2, bias=False))
    with torch.no_grad():
        linear[1].weight[0].zero_()
        linear[1].weight[1].fill_(1.0)
    x = torch.ones(2, 3, 4, 4)
    baseline = torch.zeros_like(x)
    ig = integrated_attribution(linear, x, baseline, torch.tensor([0, 1]), steps=4)
    torch.testing.assert_close(ig, x * torch.tensor([-1, 1])[:, None, None, None])

    class MarkerProbe(nn.Module):
        def __init__(self):
            super().__init__()
            self.features = nn.Identity()

        def forward(self, x):
            patch = self.features(x)[:, :, PATCH[0], PATCH[1]]
            half = PATCH_SIZE // 2
            code0 = patch[:, :, :half, half:].mean((1, 2, 3)) - patch[:, :, half:, :half].mean((1, 2, 3))
            return torch.stack([code0, -code0], dim=1)

    # Check the oracle on a fixed model whose orientation dependence is known.
    probe = MarkerProbe()
    truth = oracle(probe, frame, "fixture")
    assert truth.risky.all() and truth.aligned_correct.all() and not truth.swapped_correct.any()

    class PatchPool(nn.Sequential):
        def forward(self, x):
            features = super().forward(x)
            score = features.mean((1, 2, 3))
            return torch.stack([-score, score], dim=1)

    model = PatchPool(nn.Identity())
    x = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
    x[:, :, PATCH[0], PATCH[1]] = 1
    mass, empty = gradcam(model, model[0], x)
    torch.testing.assert_close(mass, torch.ones_like(mass))
    assert not empty.any()

    def decision_fixture():
        rows = []
        for i in range(20):
            for view in set().union(*POLICIES.values()):
                score = 100 if i == 0 else (50 if i == 1 and view == "blur" else i)
                rows.append((str(i), i % 2, "control", view, "gradcam", score))
        return pd.DataFrame(rows, columns=["image_id", "y", "arm", "view", "method", "score"])

    fixture = decision_fixture()
    thresholds = calibrate(fixture)
    assert thresholds.calibration_flag_rate.eq(0.05).all()
    assert thresholds.set_index("policy").threshold.to_dict() == {
        "nominal_only": 19.0, "context_diverse": 50.0}
    assert calibrate(fixture.assign(score=0.0)).calibration_flag_rate.eq(0.0).all()
    assert np.isnan(ratio(0, 0))
    positive = pd.DataFrame({
        "arm": ["shortcut"] * 4, "method": ["gradcam"] * 4,
        "image_id": ["a", "b", "a", "b"], "y": [0, 1, 0, 1], "risky": [True] * 4,
        "policy": ["nominal_only", "nominal_only", "context_diverse", "context_diverse"],
        "flag": [False, False, True, True],
    })
    assert paired_contrasts(positive, 19).delta_mrr.item() == 1.0
    print("Checks passed for paired rendering, the risk oracle, Grad-CAM, linear IG, calibration and paired MRR.")


def preview():
    frame = SPLIT[SPLIT.split == "dev"].groupby("y").head(1)
    images, labels = materialize(frame)
    fig, axes = plt.subplots(len(ORACLE_CONTEXTS), 2 * len(frame), figsize=(8.8, 6.6))
    for i, view in enumerate(ORACLE_CONTEXTS):
        context = contextualize(images, view)
        for j, (tag, codes) in enumerate((("aligned", labels), ("swapped", 1 - labels))):
            views = (render(context, codes).permute(0, 2, 3, 1).numpy() + 1) / 2
            for k, (image, label) in enumerate(zip(views, labels, strict=True)):
                ax = axes[i, j * len(frame) + k]
                ax.imshow(np.clip(image, 0, 1))
                ax.set_title(f"{view} / {tag}\n{CLASS_NAMES[label]}", fontsize=8)
                ax.axis("off")
    fig.tight_layout()
    fig.savefig(OUT / "interventions.png", dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def portfolio_cost(timings):
    rows = []
    for policy, views in POLICIES.items():
        by_method = timings[timings.view.isin(views)].groupby(["arm", "method"])
        total = by_method.agg(
            n_cases=("n_cases", "first"), image_views=("n_cases", "sum"),
            forward_images=("forward_images", "sum"), backward_images=("backward_images", "sum"),
            seconds=("seconds", "sum"),
        ).reset_index()
        rows.append(total.assign(policy=policy))
    result = pd.concat(rows, ignore_index=True)
    assert (result.image_views == 3 * result.n_cases).all()
    assert result.groupby(["arm", "method"]).forward_images.nunique().eq(1).all()
    assert result.groupby(["arm", "method"]).backward_images.nunique().eq(1).all()
    return result


def run_pair(arch, seed, epochs, stage=None):
    stage = stage or ("dev" if MODE == "pilot" else "test")
    checkpoint_dir = OUT / f"{arch}_{seed}"
    folder = checkpoint_dir / "development" if MODE == "study" and stage == "dev" else checkpoint_dir
    folder.mkdir(parents=True, exist_ok=MODE == "study" and stage == "test")
    frames = {split: SPLIT[SPLIT.split == split] for split in ("dev", "cal", "test")}
    dev_audit = frames["dev"].sample(frac=1, random_state=SPLIT_SEED).groupby("y").head(
        DEV_SCORE_LIMIT_PER_CLASS)
    true_rows, score_rows, time_rows = [], [], []
    for arm in ("control", "shortcut"):
        print(f"{stage} | {arch} | seed {seed} | {arm}", flush=True)
        if MODE == "study" and stage == "test":
            model, layer = make_model(arch)
            state = torch.load(checkpoint_dir / f"{arm}.pt", map_location="cpu", weights_only=True)
            model.load_state_dict(state)
            del state
            model.to(DEVICE).eval().requires_grad_(False)
        else:
            model, layer = train_model(arch, seed, arm, epochs, checkpoint_dir)
        if stage == "dev":
            true_rows.append(oracle(model, frames["dev"], arm))
            scores, times = score_views(model, layer, dev_audit, arm)
        else:
            if arm == "control":
                cal_scores, cal_times = score_views(model, layer, frames["cal"], arm)
                cal_scores.to_csv(folder / "cal_scores.csv", index=False)
                portfolio_cost(cal_times).to_csv(folder / "cal_budget.csv", index=False)
                thresholds = calibrate(cal_scores)
                thresholds.to_csv(folder / "thresholds.csv", index=False)
            true_rows.append(oracle(model, frames["test"], arm))
            scores, times = score_views(model, layer, frames["test"], arm)
        score_rows.append(scores)
        time_rows.append(times)
        del model, layer
    truth, scores, times = (pd.concat(parts, ignore_index=True)
                            for parts in (true_rows, score_rows, time_rows))
    truth.to_csv(folder / "oracle_cases.csv", index=False)
    oracle_budget = truth.groupby(["arm", "view"]).size().rename("n_cases").reset_index()
    oracle_budget.assign(forward_images=3 * oracle_budget.n_cases, backward_images=0).to_csv(
        folder / "oracle_budget.csv", index=False)
    scores.to_csv(folder / "view_scores.csv", index=False)
    times.to_csv(folder / "view_budget.csv", index=False)
    portfolio_cost(times).to_csv(folder / "portfolio_budget.csv", index=False)
    oracle_summary = summarize_oracle(truth)
    oracle_summary.to_csv(folder / "oracle_summary.csv", index=False)
    quality = scores.groupby(["arm", "method", "view"]).agg(
        empty_map_rate=("empty_map", "mean"),
        ig_median=("ig_residual", "median"),
        ig_p90=("ig_residual", lambda s: s.quantile(0.90)),
    ).reset_index()
    quality.to_csv(folder / "attribution_diagnostics.csv", index=False)
    print(oracle_summary.to_string(index=False))
    if stage == "dev":
        shortcut = stress_risk(truth[truth.arm == "shortcut"])
        control = stress_risk(truth[truth.arm == "control"])
        shortcut_nominal = truth[(truth.arm == "shortcut") & (truth.view == "nominal")].set_index(
            "image_id").risky
        stress_only = shortcut & ~shortcut_nominal
        accuracy = oracle_summary[oracle_summary.view == "nominal"].set_index("arm")
        ig_p90 = float(quality.ig_p90.max())
        checks = {
            "enough_stress_risk": bool(shortcut.sum() >= DEV_MIN_RISK),
            "enough_stress_only_risk": bool(stress_only.sum() >= DEV_MIN_STRESS_ONLY_RISK),
            "shortcut_nominal_aligned_accuracy": bool(
                accuracy.loc["shortcut", "aligned_accuracy"] >= DEV_MIN_ALIGNED_ACCURACY),
            "control_nominal_neutral_accuracy": bool(
                accuracy.loc["control", "neutral_accuracy"] >= DEV_MIN_CONTROL_NOMINAL_ACCURACY),
            "control_stress_risk": bool(control.mean() <= DEV_MAX_CONTROL_STRESS_RISK),
            "ig_p90_residual": ig_p90 <= DEV_MAX_IG_P90_RESIDUAL,
        }
        gate = {"checks": checks, "all_passed": all(checks.values()),
                "shortcut_stress_risk_n": int(shortcut.sum()), "control_stress_risk_n": int(control.sum()),
                "shortcut_stress_only_risk_n": int(stress_only.sum()),
                "n_dev": len(shortcut), "worst_ig_p90": ig_p90}
        save_json(folder / "pilot_gate.json", gate)
        print(json.dumps(gate, indent=2))
        return gate

    decisions = make_decisions(scores, truth, thresholds)
    decisions.to_csv(folder / "decisions.csv", index=False)
    summary = summarize_decisions(decisions)
    contrast = paired_contrasts(decisions, seed + SPLIT_SEED)
    summary.to_csv(folder / "metrics.csv", index=False)
    contrast.to_csv(folder / "paired_contrasts.csv", index=False)
    secondary = decisions.assign(risky=decisions.risky & ~decisions.nominal_risky)
    paired_contrasts(secondary, seed + SPLIT_SEED).to_csv(
        folder / "stress_only_contrasts.csv", index=False)
    risk_sensitivity(decisions, truth, seed + SPLIT_SEED).to_csv(
        folder / "risk_sensitivity.csv", index=False)
    pd.concat([summarize_decisions(decisions[decisions.y == label]).assign(y=label)
               for label in CLASSES.values()], ignore_index=True).to_csv(
        folder / "class_metrics.csv", index=False)
    return summary.assign(architecture=arch, seed=seed), contrast.assign(architecture=arch, seed=seed)

## Run and export

The feasibility gate uses every development case for accuracy and risk and checks IG
completeness on a fixed label-stratified subset. All six checkpoint pairs run through
this gate before the notebook opens calibration and test. A failed gate produces an
export for development review.

The results ZIP includes configuration, split and marker manifests, per-case scores,
outcomes, decisions, budget logs, and checkpoint checksums. Checkpoint files stay in
the Colab run folder while the session is active.


In [ ]:
configuration = {k: v for k, v in list(globals().items()) if k in (
    "MODE", "STUDY_FROZEN", "PILOT_ARCH", "PILOT_SEED", "PILOT_EPOCHS", "STUDY_EPOCHS",
    "STUDY_ARCHS", "STUDY_SEEDS", "SPLIT_SEED", "IMAGE_SIZE", "PATCH_TOP", "PATCH_LEFT",
    "PATCH_SIZE", "ALPHA", "TRAIN_ALIGNMENT", "BLUR_RADIUS", "OCCLUSION_SIZE", "MARGIN_DROP",
    "IG_STEPS", "IG_CHUNK", "AUDIT_BATCH", "TRAIN_BATCH", "LEARNING_RATE", "WEIGHT_DECAY",
    "CAL_FALSE_FLAG_RATE", "N_BOOT", "N_DEV_PER_CLASS", "N_CAL_PER_CLASS", "CLASSES", "CLASS_NAMES", "URL",
    "ARCHIVE_SHA256",
    "PROTOCOL_VERSION", "REFRESH_BATCH_NORM", "POLICIES", "ORACLE_CONTEXTS", "METHODS", "DEV_SCORE_LIMIT_PER_CLASS",
    "DEV_MIN_RISK", "DEV_MIN_STRESS_ONLY_RISK", "DEV_MIN_ALIGNED_ACCURACY", "DEV_MIN_CONTROL_NOMINAL_ACCURACY",
    "DEV_MAX_CONTROL_STRESS_RISK", "DEV_MAX_IG_P90_RESIDUAL",
)}
configuration.update(archive_sha256=archive_hash, device=str(DEVICE), platform=platform.platform(),
                     python=platform.python_version(), torch=torch.__version__,
                     torchvision=torchvision.__version__, cuda=torch.version.cuda,
                     cudnn=torch.backends.cudnn.version(), numpy=np.__version__, pandas=pd.__version__,
                     pillow=pillow_version, gpu=torch.cuda.get_device_name() if DEVICE.type == "cuda" else None)
save_json(OUT / "run_config.json", configuration)

implementation_checks()
preview()
if MODE == "pilot":
    run_pair(PILOT_ARCH, PILOT_SEED, PILOT_EPOCHS)
else:
    planned = [(PILOT_ARCH, PILOT_SEED, 1)] if MODE == "smoke" else [
        (arch, seed, STUDY_EPOCHS) for arch in STUDY_ARCHS for seed in STUDY_SEEDS]
    if MODE == "study":
        gates = [{"architecture": arch, "seed": seed, **run_pair(arch, seed, epochs, stage="dev")}
                 for arch, seed, epochs in planned]
        save_json(OUT / "study_development_gates.json", gates)
    else:
        gates = []
    if all(gate["all_passed"] for gate in gates):
        results = [run_pair(arch, seed, epochs, stage="test") for arch, seed, epochs in planned]
        for filename, tables in zip(("study_metrics.csv", "study_paired_contrasts.csv"),
                                    zip(*results, strict=True), strict=True):
            table = pd.concat(tables, ignore_index=True)
            table.to_csv(OUT / filename, index=False)
            print(table.to_string(index=False))
    else:
        print("Development checks require review. This export contains the development results.")
if MODE == "study":
    checkpoints = sorted(OUT.glob("*/*.pt"))
    save_json(OUT / "checkpoint_manifest.json", [
        {"path": path.relative_to(OUT).as_posix(), "sha256": file_sha256(path),
         "bytes": path.stat().st_size} for path in checkpoints
    ])
    ZIP = OUT.with_name(f"{OUT.name}_results.zip")
    with zipfile.ZipFile(ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUT.rglob("*")):
            if path.is_file() and path.suffix != ".pt":
                archive.write(path, path.relative_to(OUT).as_posix())
    print(f"Checkpoints are available in {OUT}")
else:
    ZIP = Path(shutil.make_archive(str(OUT), "zip", root_dir=OUT))
print(f"Archive ready: {ZIP}")

In [ ]:
# Run this cell again if the browser blocks the first download.
try:
    from google.colab import files
except ImportError:
    print(ZIP)
else:
    files.download(str(ZIP))